# Phase 2 ESN Regression Baseline Tuning

This notebook evaluates a bounded, library-backed Echo State Network regression baseline for the Phase 2 volatility-forecasting task.

Purpose:

- reuse the useful ReservoirPy ESN infrastructure from earlier binary stress-classification work;
- adapt it to continuous realized-volatility targets;
- compare PCA-compressed ESN baselines against the classical HAR/Ridge/ElasticNet floor;
- avoid an extensive hyperparameter campaign.


## 1. Imports and working directory

In [ ]:
from pathlib import Path
import os

if Path.cwd().name == "notebooks":
    os.chdir("..")

import pandas as pd

from qpitome_qrc.data.features import FEATURE_COLUMNS, make_sequence_arrays
from qpitome_qrc.data.loaders import load_phase2_volatility_data
from qpitome_qrc.data.pca import fit_transform_pca_splits_train_only
from qpitome_qrc.data.splits import chronological_tabular_split
from qpitome_qrc.baselines.esn_regression import (
    ESNRegressionConfig,
    aggregate_esn_regression_seeds,
    fit_single_esn_regressor,
    summarize_esn_regression_run,
)

## 2. Load data and build chronological splits

In [ ]:
df = load_phase2_volatility_data()
splits = chronological_tabular_split(df)

{name: split.shape for name, split in splits.items()}

## 3. Train-only PCA diagnostics

In [ ]:
target = "future_rv_20d"
pca6 = fit_transform_pca_splits_train_only(
    splits,
    feature_columns=FEATURE_COLUMNS,
    target_columns=[target],
    n_components=6,
    prefix="pca6",
)
pca6.explained_variance

## 4. Build PCA-6 sequence splits

In [ ]:
def build_sequence_splits(transformed_splits, feature_columns, target_column, lookback):
    return {
        name: make_sequence_arrays(
            split,
            feature_columns=feature_columns,
            target_column=target_column,
            lookback=lookback,
        )
        for name, split in transformed_splits.items()
    }

seq_len = 20
sequence_splits = build_sequence_splits(
    pca6.splits,
    pca6.feature_columns,
    target,
    lookback=seq_len,
)

{name: (X.shape, y.shape) for name, (X, y, dates) in sequence_splits.items()}

## 5. Run bounded ESN regression configurations

In [ ]:
configs = [
    ESNRegressionConfig(units=300, spectral_radius=0.7, leak_rate=0.5, reservoir_connectivity=0.1, ridge_alpha=1.0, seed=seed)
    for seed in (1, 2, 3)
]

rows = []
for config in configs:
    print(config)
    result = fit_single_esn_regressor(
        sequence_splits,
        config=config,
        target=target,
        feature_set="pca6",
        seq_len=seq_len,
    )
    rows.append(summarize_esn_regression_run(result))

esn_runs = pd.DataFrame(rows)
esn_runs.sort_values("val_rmse")

## 6. Aggregate seed stability

In [ ]:
esn_aggregate = aggregate_esn_regression_seeds(esn_runs)
esn_aggregate

## 7. Save exploratory results

In [ ]:
out_dir = Path("results/tables")
out_dir.mkdir(parents=True, exist_ok=True)

esn_runs.to_csv(out_dir / "phase2_esn_regression_runs.csv", index=False)
esn_aggregate.to_csv(out_dir / "phase2_esn_regression_seed_aggregate.csv", index=False)
pca6.explained_variance.to_csv(out_dir / "phase2_pca6_explained_variance.csv", index=False)

out_dir